# **RESULT AFTER PREPROCESSING**


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from functools import reduce
from scipy.stats import gaussian_kde, pearsonr
from matplotlib import gridspec
from matplotlib.lines import Line2D
from scipy.interpolate import interp1d
import pickle


In [ ]:
EDA_FREQ = 15  # Updated frequency
BVP_FREQ = 64

def extract_freq(signal, device):
    new_sampling_rates = {
        "emotibit": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ, "Pr": BVP_FREQ, "Pi": BVP_FREQ},
        "shimmer": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ}
    }

    device_lower = device.lower()
    if "emotibit" in device_lower and signal in new_sampling_rates["emotibit"]:
        return new_sampling_rates["emotibit"][signal]
    elif "shimmer" in device_lower and signal in new_sampling_rates["shimmer"]:
        return new_sampling_rates["shimmer"][signal]
    else:
        return None

In [ ]:
def load_events(user):
    events = pd.read_csv(
        f"data/Stamps/{user}.txt", header=None, names=['Event', 'Timestamp']
    )

    return events

## Scatter and density plot

In [ ]:
def plot_scatter_pearson(
    data,
    signal,
    device_names,
    comparisons,
    comparisons_tick,
    colors,
    output_pdf="output.pdf",
    task=None
):

    try:
        users_list = list(data[device_names[0]].keys())
    except Exception as e:
        print(f"[WARNING] Could not get users: {e}")
        return

    with PdfPages(output_pdf) as pdf:

        # Arbitrary user used only to inspect the nested processing structure
        user_id = users_list[0]
        signals_ref = [data[d][user_id][signal] for d in device_names]

        def plot_users(devices_ref, techniques=["original"]):

            try:
                if "df" in devices_ref[0]:
                    return
            except Exception:
                pass

            for technique in devices_ref[0]:
                try:

                    if technique == "original":

                        # Former configuration used for phase == "second"
                        fig = plt.figure(figsize=(8, 4), dpi=200)
                        gs = gridspec.GridSpec(
                            1,
                            2,
                            width_ratios=[5, 1],
                            wspace=0.05
                        )

                        ax = fig.add_subplot(gs[0])
                        ax_density = fig.add_subplot(gs[1], sharey=ax)

                        ax_density.yaxis.set_visible(False)
                        ax_density.set_xticks([])
                        ax_density.set_frame_on(False)

                        user_results = {}

                        for user in users_list:

                            user_results[user] = []

                            for idx, (dev1, dev2) in enumerate(comparisons):

                                try:
                                    df1 = reduce(
                                        lambda d, k: d[k],
                                        techniques[1:],
                                        data[dev1][user][signal]
                                    )["original"]["df"]

                                    df2 = reduce(
                                        lambda d, k: d[k],
                                        techniques[1:],
                                        data[dev2][user][signal]
                                    )["original"]["df"]

                                    # Restrict analysis to a specific experimental task
                                    if task:
                                        events = load_events(user)

                                        t0 = events.loc[
                                            events["Event"] == task[0],
                                            "Timestamp"
                                        ].values[0]

                                        t1 = events.loc[
                                            events["Event"] == task[1],
                                            "Timestamp"
                                        ].values[0]

                                        df1 = df1[
                                            (df1["Timestamp"] >= t0) &
                                            (df1["Timestamp"] <= t1)
                                        ]

                                        df2 = df2[
                                            (df2["Timestamp"] >= t0) &
                                            (df2["Timestamp"] <= t1)
                                        ]

                                    if df1.empty or df2.empty:
                                        user_results[user].append(
                                            (np.nan, np.nan)
                                        )
                                        continue

                                    if (
                                        "Timestamp" not in df1.columns or
                                        "Timestamp" not in df2.columns
                                    ):
                                        user_results[user].append(
                                            (np.nan, np.nan)
                                        )
                                        continue

                                    # Find common temporal interval
                                    min_time = max(
                                        df1["Timestamp"].min(),
                                        df2["Timestamp"].min()
                                    )

                                    max_time = min(
                                        df1["Timestamp"].max(),
                                        df2["Timestamp"].max()
                                    )

                                    if min_time >= max_time:
                                        user_results[user].append(
                                            (np.nan, np.nan)
                                        )
                                        continue

                                    # Determine number of samples used
                                    # for temporal alignment
                                    fs = extract_freq(signal, dev1)

                                    if fs is not None:

                                        num_points = int(
                                            (max_time - min_time) * fs
                                        )

                                        if num_points < 2:
                                            user_results[user].append(
                                                (np.nan, np.nan)
                                            )
                                            continue

                                    else:

                                        len1 = (
                                            (df1["Timestamp"] >= min_time) &
                                            (df1["Timestamp"] <= max_time)
                                        ).sum()

                                        len2 = (
                                            (df2["Timestamp"] >= min_time) &
                                            (df2["Timestamp"] <= max_time)
                                        ).sum()

                                        num_points = min(len1, len2)

                                        if num_points < 2:
                                            user_results[user].append(
                                                (np.nan, np.nan)
                                            )
                                            continue

                                    # Align both signals on a common time axis
                                    common_time = np.linspace(
                                        min_time,
                                        max_time,
                                        num_points
                                    )

                                    f1 = interp1d(
                                        df1["Timestamp"],
                                        df1[df1.columns[1]],
                                        kind="nearest",
                                        bounds_error=False,
                                        fill_value="extrapolate"
                                    )

                                    f2 = interp1d(
                                        df2["Timestamp"],
                                        df2[df2.columns[1]],
                                        kind="nearest",
                                        bounds_error=False,
                                        fill_value="extrapolate"
                                    )

                                    aligned1 = f1(common_time)
                                    aligned2 = f2(common_time)

                                    if (
                                        np.all(np.isnan(aligned1)) or
                                        np.all(np.isnan(aligned2))
                                    ):
                                        user_results[user].append(
                                            (np.nan, np.nan)
                                        )
                                        continue

                                    r, p = pearsonr(
                                        aligned1,
                                        aligned2
                                    )

                                    user_results[user].append((r, p))

                                except Exception as e:
                                    print(
                                        f"[ERROR] Error processing "
                                        f"{dev1}-{dev2} for user "
                                        f"{user}: {e}"
                                    )

                                    user_results[user].append(
                                        (np.nan, np.nan)
                                    )

                        # Scatter plot
                        for user, values in user_results.items():

                            x_vals = []
                            y_vals = []

                            for idx, (r, p) in enumerate(values):

                                if np.isnan(r):
                                    continue

                                jitter = (
                                    np.random.rand() - 0.5
                                ) * 0.08

                                x_pos = idx + jitter

                                x_vals.append(x_pos)
                                y_vals.append(r)

                                marker = "o" if p <= 0.05 else "X"

                                ax.scatter(
                                    x_pos,
                                    r,
                                    s=60,
                                    color=colors[
                                        idx % len(colors)
                                    ],
                                    edgecolors="k",
                                    alpha=1,
                                    marker=marker,
                                    zorder=2
                                )

                            if len(x_vals) > 1:
                                ax.plot(
                                    x_vals,
                                    y_vals,
                                    color="gray",
                                    alpha=0.7,
                                    linewidth=1,
                                    zorder=1
                                )

                        # Density plots
                        try:
                            y_density = np.linspace(-1, 1, 500)

                            for idx, (dev1, dev2) in enumerate(
                                comparisons
                            ):

                                corr_vals = [
                                    user_results[u][idx][0]
                                    for u in users_list
                                    if not np.isnan(
                                        user_results[u][idx][0]
                                    )
                                ]

                                if len(corr_vals) > 1:

                                    kde = gaussian_kde(corr_vals)

                                    density = kde(y_density)

                                    density = (
                                        density /
                                        density.max() *
                                        0.3
                                    )

                                    ax_density.fill_betweenx(
                                        y_density,
                                        0,
                                        density,
                                        color=colors[
                                            idx % len(colors)
                                        ],
                                        alpha=0.3
                                    )

                        except Exception as e:
                            print(
                                f"[ERROR] Error generating density: {e}"
                            )

                        # Legend and formatting
                        legend_elements = [
                            Line2D(
                                [0],
                                [0],
                                marker="o",
                                color="k",
                                label="p ≤ 0.05",
                                markersize=8,
                                linestyle=""
                            ),
                            Line2D(
                                [0],
                                [0],
                                marker="X",
                                color="k",
                                label="p > 0.05",
                                markersize=8,
                                linestyle=""
                            )
                        ]

                        ax.legend(
                            handles=legend_elements,
                            loc="lower right",
                            fontsize=10
                        )

                        ax.set_xticks(
                            range(len(comparisons))
                        )

                        ax.set_xticklabels(
                            comparisons_tick,
                            fontsize=12
                        )

                        ax.set_ylim(-1, 1)

                        ax.set_ylabel(
                            "Pearson Correlation (R)",
                            fontsize=12
                        )

                        ax.spines["top"].set_visible(False)
                        ax.spines["right"].set_visible(False)
                        ax.grid(False)

                        pdf.savefig(
                            fig,
                            dpi=200,
                            bbox_inches="tight"
                        )

                        plt.close(fig)

                        # Summary page
                        fig_summary = plt.figure(
                            figsize=(8, 6),
                            dpi=200
                        )

                        text_lines = [
                            "Correlation Summary - "
                            f"Technique: {' -> '.join(techniques)}\n"
                        ]

                        for idx, (dev1, dev2) in enumerate(
                            comparisons
                        ):

                            vals = [
                                user_results[u][idx][0]
                                for u in users_list
                                if not np.isnan(
                                    user_results[u][idx][0]
                                )
                            ]

                            pvals = [
                                user_results[u][idx][1]
                                for u in users_list
                                if not np.isnan(
                                    user_results[u][idx][1]
                                )
                            ]

                            if not vals:
                                continue

                            mean_r = np.mean(vals)
                            std_r = np.std(vals)
                            min_r = np.min(vals)
                            max_r = np.max(vals)

                            mean_p = np.mean(pvals)
                            min_p = np.min(pvals)
                            max_p = np.max(pvals)

                            nonsig_count = sum(
                                p > 0.05
                                for p in pvals
                            )

                            text_lines.append(
                                f"{dev1.capitalize()} vs "
                                f"{dev2.capitalize()}:\n"
                                f"  Mean R: {mean_r:.3f}\n"
                                f"  Std R: {std_r:.3f}\n"
                                f"  Min R: {min_r:.3f}, "
                                f"Max R: {max_r:.3f}\n"
                                f"  Mean p-value: "
                                f"{mean_p:.3e}\n"
                                f"  Min p-value: "
                                f"{min_p:.3e}, "
                                f"Max p-value: "
                                f"{max_p:.3e}\n"
                                f"  Non-significant count "
                                f"(p > 0.05): "
                                f"{nonsig_count}\n"
                            )

                        fig_summary.text(
                            0.1,
                            0.5,
                            "\n".join(text_lines),
                            fontsize=12,
                            va="center"
                        )

                        pdf.savefig(
                            fig_summary,
                            dpi=200,
                            bbox_inches="tight"
                        )

                        plt.close(fig_summary)

                    else:
                        plot_users(
                            [
                                ref[technique]
                                for ref in devices_ref
                            ],
                            techniques + [technique]
                        )

                except Exception as e:
                    print(
                        f"[ERROR] Error in technique "
                        f"{technique}: {e}"
                    )

        plot_users(signals_ref)

    print(f"[OK] PDF generated: {output_pdf}")

## Behaviour plot line

In [ ]:
import os
from functools import reduce

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def _event_timestamp(events, name):

    values = events.loc[
        events["Event"] == name,
        "Timestamp"
    ].values

    if len(values) == 0:
        raise KeyError(
            f"Missing event '{name}'"
        )

    return float(values[0])


def _representative_event_durations(
    users,
    tasks
):
    """
    Median duration of each experimental task
    across participants.
    """

    durations_by_task = {}

    for start_event, end_event, label in tasks:

        durations = []

        for user in users:

            try:
                events = load_events(user)

                t0 = _event_timestamp(
                    events,
                    start_event
                )

                t1 = _event_timestamp(
                    events,
                    end_event
                )

                if (
                    np.isfinite(t0)
                    and np.isfinite(t1)
                    and t1 > t0
                ):
                    durations.append(
                        t1 - t0
                    )

            except Exception:
                continue

        if not durations:
            raise ValueError(
                f"No valid durations found for {label}"
            )

        durations_by_task[label] = float(
            np.median(durations)
        )

    return durations_by_task


def _make_real_time_layout(
    tasks,
    median_durations,
    n_means=8
):
    """
    Place the n_means values of each task on an
    x-axis proportional to its median duration.
    """

    x_values = []
    task_bands = []

    cursor = 0.0

    for _, _, label in tasks:

        duration = median_durations[label]

        edges = np.linspace(
            cursor,
            cursor + duration,
            n_means + 1
        )

        centers = (
            edges[:-1] + edges[1:]
        ) / 2

        x_values.extend(centers)

        task_bands.append(
            (
                label,
                cursor,
                cursor + duration
            )
        )

        cursor += duration

    return (
        np.asarray(x_values, dtype=float),
        task_bands
    )


def _participant_behaviour(
    data,
    user,
    device,
    signal,
    tasks,
    technique_path,
    n_means=8
):
    """
    Calculate the same segment-wise values used
    in the previous n_means analysis.
    """

    events = load_events(user)

    df = reduce(
        lambda d, k: d[k],
        technique_path,
        data[device][user][signal]
    )["original"]["df"]

    y_values = []

    for start_event, end_event, _ in tasks:

        t_start = _event_timestamp(
            events,
            start_event
        )

        t_end = _event_timestamp(
            events,
            end_event
        )

        segment_length = (
            t_end - t_start
        ) / n_means

        for i in range(n_means):

            seg_start = (
                t_start
                + i * segment_length
            )

            seg_end = (
                t_start
                + (i + 1) * segment_length
            )

            df_segment = df[
                (df["Timestamp"] >= seg_start)
                &
                (df["Timestamp"] < seg_end)
            ]

            if df_segment.empty:

                y_values.append(
                    np.nan
                )

            else:

                value = df_segment[
                    df_segment.columns[1]
                ].mean()

                y_values.append(
                    np.log(value + 1)
                )

    return np.asarray(
        y_values,
        dtype=float
    )


def _bootstrap_mean_ci(
    values,
    confidence=0.95,
    n_boot=5000,
    seed=2026
):
    """
    Pointwise bootstrap confidence interval
    of the participant mean.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    mean = np.nanmean(
        values,
        axis=0
    )

    n_valid = np.sum(
        np.isfinite(values),
        axis=0
    )

    ci_low = np.full(
        values.shape[1],
        np.nan
    )

    ci_high = np.full(
        values.shape[1],
        np.nan
    )

    rng = np.random.default_rng(seed)

    alpha = (
        1.0 - confidence
    ) / 2.0

    for index in range(
        values.shape[1]
    ):

        point_values = values[:, index]

        point_values = point_values[
            np.isfinite(point_values)
        ]

        if len(point_values) == 0:
            continue

        if len(point_values) == 1:

            ci_low[index] = (
                point_values[0]
            )

            ci_high[index] = (
                point_values[0]
            )

            continue

        bootstrap_indices = rng.integers(
            0,
            len(point_values),
            size=(
                n_boot,
                len(point_values)
            )
        )

        bootstrap_means = (
            point_values[
                bootstrap_indices
            ].mean(axis=1)
        )

        ci_low[index], ci_high[index] = (
            np.quantile(
                bootstrap_means,
                [
                    alpha,
                    1.0 - alpha
                ]
            )
        )

    return (
        mean,
        ci_low,
        ci_high,
        n_valid
    )


def _interpolate_internal_gaps(
    y
):
    """
    Interpolate internal NaN gaps of participant
    trajectories.

    Original values are retained for the group
    mean and bootstrap confidence interval.
    """

    y = np.asarray(
        y,
        dtype=float
    ).copy()

    finite = np.isfinite(y)

    if finite.sum() < 2:
        return y

    indices = np.arange(len(y))

    first = indices[finite][0]
    last = indices[finite][-1]

    internal = (
        ~finite
        &
        (indices > first)
        &
        (indices < last)
    )

    y[internal] = np.interp(
        indices[internal],
        indices[finite],
        y[finite]
    )

    return y

In [ ]:

def plot_behaviour(
    data,
    device_names,
    tasks,
    output_pdf,
    output_png=None,
    n_means=8,
    n_boot=5000,
    bootstrap_seed=2026
):

    signals = (
        "Gsr",
        "Hr"
    )

    users = list(
        data[
            device_names[0]
        ].keys()
    )

    # Processing branches used in the manuscript
    technique_paths = {
        "Gsr": (
            "IQR",
            "gaussian"
        ),
        "Hr": (
            "IQR",
            "butterworth"
        ),
    }

    # Real-time X-axis
    median_durations = (
        _representative_event_durations(
            users,
            tasks
        )
    )

    x_seconds, task_bands = (
        _make_real_time_layout(
            tasks,
            median_durations,
            n_means=n_means
        )
    )

    x_minutes = (
        x_seconds / 60.0
    )

    task_bands_minutes = [
        (
            label,
            start / 60.0,
            end / 60.0
        )
        for label, start, end
        in task_bands
    ]

    total_minutes = (
        task_bands_minutes[-1][2]
    )

    # Event colours
    event_colors = [
        "#b9acd2",  # Baseline
        "#e2c65b",  # Squat test
        "#82b5d5",  # Relax 1
        "#dfaa81",  # Video 1
        "#88c39c",  # Relax 2
        "#c89a5c",  # Video 2
    ]

    # Calculate participant curves
    curves = {}

    row_values = {
        signal: []
        for signal in signals
    }

    for signal in signals:

        for device in device_names:

            device_curves = []

            for user in users:

                try:

                    y = _participant_behaviour(
                        data=data,
                        user=user,
                        device=device,
                        signal=signal,
                        tasks=tasks,
                        technique_path=(
                            technique_paths[
                                signal
                            ]
                        ),
                        n_means=n_means
                    )

                    device_curves.append(
                        (
                            user,
                            y
                        )
                    )

                    row_values[
                        signal
                    ].extend(
                        y[
                            np.isfinite(y)
                        ].tolist()
                    )

                except Exception as e:

                    print(
                        f"[WARNING] {signal} - "
                        f"{device} - "
                        f"user {user}: {e}"
                    )

            curves[
                (
                    signal,
                    device
                )
            ] = device_curves

    # Shared Y limits for each row
    row_limits = {}

    for signal in signals:

        values = np.asarray(
            row_values[signal],
            dtype=float
        )

        ymin = np.nanmin(values)
        ymax = np.nanmax(values)

        if signal == "Hr":
            padding = 0.035

        else:
            padding = max(
                0.05,
                0.04 * (
                    ymax - ymin
                )
            )

        row_limits[signal] = (
            ymin - padding,
            ymax + padding
        )

    # Figure
    fig, axes = plt.subplots(
        2,
        4,
        figsize=(15.0, 5.6),
        dpi=200,
        sharex=True
    )

    plt.subplots_adjust(
        left=0.055,
        right=0.93,
        top=0.90,
        bottom=0.12,
        wspace=0.10,
        hspace=0.12
    )

    # Panels
    for row, signal in enumerate(
        signals
    ):

        for column, device in enumerate(
            device_names
        ):

            ax = axes[
                row,
                column
            ]

            # Task backgrounds
            for i, (
                label,
                start,
                end
            ) in enumerate(
                task_bands_minutes
            ):

                ax.axvspan(
                    start,
                    end,
                    facecolor=(
                        event_colors[i]
                    ),
                    alpha=0.18,
                    linewidth=0,
                    zorder=0
                )

                if i > 0:

                    ax.axvline(
                        start,
                        color="0.55",
                        linewidth=0.8,
                        linestyle="--",
                        alpha=0.65,
                        zorder=1
                    )

            device_curves = curves[
                (
                    signal,
                    device
                )
            ]

            # Individual participants
            for _, y in device_curves:

                y_display = (
                    _interpolate_internal_gaps(
                        y
                    )
                )

                ax.plot(
                    x_minutes,
                    y_display,
                    color="0.55",
                    linewidth=0.90,
                    alpha=0.30,
                    zorder=2
                )

            # Group mean + bootstrap CI
            if device_curves:

                Y = np.vstack(
                    [
                        y
                        for _, y
                        in device_curves
                    ]
                )

                (
                    mean_y,
                    ci_low,
                    ci_high,
                    valid_n
                ) = _bootstrap_mean_ci(
                    Y,
                    confidence=0.95,
                    n_boot=n_boot,
                    seed=bootstrap_seed
                )

                ax.fill_between(
                    x_minutes,
                    ci_low,
                    ci_high,
                    facecolor="#ef9a93",
                    alpha=0.25,
                    linewidth=0,
                    zorder=2.5
                )

                ax.plot(
                    x_minutes,
                    mean_y,
                    color="#d8241f",
                    linewidth=2.20,
                    zorder=3
                )

                n_panel = sum(
                    np.any(
                        np.isfinite(y)
                    )
                    for _, y
                    in device_curves
                )

                ax.text(
                    0.02,
                    0.97,
                    f"n = {n_panel}",
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=9,
                    color="0.35"
                )

            # Axis configuration
            ax.set_xlim(
                0,
                total_minutes
            )

            ax.set_ylim(
                *row_limits[
                    signal
                ]
            )

            ax.tick_params(
                axis="both",
                labelsize=9,
                direction="out",
                length=3.5
            )

            if column == 0:

                unit = (
                    "µS"
                    if signal == "Gsr"
                    else "bpm"
                )

                ax.set_ylabel(
                    f"log(mean + 1 ({unit}))",
                    fontsize=11
                )

            else:

                ax.tick_params(
                    axis="y",
                    labelleft=False
                )

            if row == 1:

                ax.set_xlabel(
                    "Time (min)",
                    fontsize=10
                )

            else:

                ax.tick_params(
                    axis="x",
                    labelbottom=False
                )

    # Explanation at the top
    fig.text(
        0.62,
        0.965,
        "Grey: participants",
        fontsize=9.5,
        color="0.42"
    )

    fig.text(
        0.735,
        0.965,
        "Red: group mean",
        fontsize=9.5,
        color="#d8241f"
    )

    fig.text(
        0.835,
        0.965,
        "Red band:",
        fontsize=9.5,
        color="#ef8f8a"
    )

    fig.text(
        0.895,
        0.965,
        "95% bootstrap CI",
        fontsize=9.5,
        color="0.1"
    )

    # Vertical event legend
    legend_ax = fig.add_axes(
        [
            0.935,
            0.12,
            0.06,
            0.78
        ]
    )

    legend_ax.axis("off")

    n_events = len(
        task_bands_minutes
    )

    for i, (
        label,
        _,
        _
    ) in enumerate(
        task_bands_minutes
    ):

        y = (
            1.0
            - (i + 0.5)
            / n_events
        )

        legend_ax.add_patch(
            Rectangle(
                (
                    0.02,
                    y - 0.035
                ),
                0.18,
                0.07,
                facecolor=(
                    event_colors[i]
                ),
                edgecolor="none",
                transform=(
                    legend_ax.transAxes
                )
            )
        )

        legend_ax.text(
            0.48,
            y,
            label,
            rotation=-90,
            va="center",
            ha="center",
            fontsize=8.5,
            color="0.12",
            transform=(
                legend_ax.transAxes
            )
        )

    # Save
    os.makedirs(
        os.path.dirname(
            output_pdf
        ) or ".",
        exist_ok=True
    )

    fig.savefig(
        output_pdf,
        facecolor="white",
        bbox_inches="tight"
    )

    if output_png:

        fig.savefig(
            output_png,
            dpi=300,
            facecolor="white",
            bbox_inches="tight"
        )

    plt.close(fig)

    print(
        f"[OK] Saved: {output_pdf}"
    )

    if output_png:
        print(
            f"[OK] Saved: {output_png}"
        )

## Tabla

In [ ]:
import numpy as np
from functools import reduce
from scipy.stats import pearsonr
from scipy.interpolate import interp1d


def get_table_data(
    data,
    signal,
    device_names,
    comparisons,
    comparisons_name,
    crop=None
):

    techniques_comparisons = {}

    try:
        users_list = list(data[device_names[0]].keys())
    except Exception as e:
        print(f"[WARNING] Could not get users: {e}")
        return

    user_id = users_list[0]

    signals_ref = [
        data[d][user_id][signal]
        for d in device_names
    ]

    def calculate_corrs(devices_ref, techniques=["original"]):

        try:
            if "df" in devices_ref[0]:
                return
        except Exception:
            pass

        for technique in devices_ref[0]:

            try:
                if technique == "original":

                    user_results = {}

                    for user in users_list:

                        user_results[user] = []

                        for idx, (dev1, dev2) in enumerate(comparisons):

                            try:
                                df1 = reduce(
                                    lambda d, k: d[k],
                                    techniques[1:],
                                    data[dev1][user][signal]
                                )["original"]["df"]

                                df2 = reduce(
                                    lambda d, k: d[k],
                                    techniques[1:],
                                    data[dev2][user][signal]
                                )["original"]["df"]

                                # Restrict analysis to a specific
                                # experimental interval, if requested
                                if crop:
                                    events = load_events(user)

                                    t0 = events.loc[
                                        events["Event"] == crop[0],
                                        "Timestamp"
                                    ].values[0]

                                    t1 = events.loc[
                                        events["Event"] == crop[1],
                                        "Timestamp"
                                    ].values[0]

                                    df1 = df1[
                                        (df1["Timestamp"] >= t0)
                                        &
                                        (df1["Timestamp"] <= t1)
                                    ]

                                    df2 = df2[
                                        (df2["Timestamp"] >= t0)
                                        &
                                        (df2["Timestamp"] <= t1)
                                    ]

                                if df1.empty or df2.empty:
                                    user_results[user].append(
                                        (np.nan, np.nan)
                                    )
                                    continue

                                if (
                                    "Timestamp" not in df1.columns
                                    or
                                    "Timestamp" not in df2.columns
                                ):
                                    user_results[user].append(
                                        (np.nan, np.nan)
                                    )
                                    continue

                                min_time = max(
                                    df1["Timestamp"].min(),
                                    df2["Timestamp"].min()
                                )

                                max_time = min(
                                    df1["Timestamp"].max(),
                                    df2["Timestamp"].max()
                                )

                                if min_time >= max_time:
                                    user_results[user].append(
                                        (np.nan, np.nan)
                                    )
                                    continue

                                fs = extract_freq(
                                    signal,
                                    dev1
                                )

                                if fs is not None:

                                    num_points = int(
                                        (max_time - min_time) * fs
                                    )

                                    if num_points < 2:
                                        user_results[user].append(
                                            (np.nan, np.nan)
                                        )
                                        continue

                                else:

                                    len1 = (
                                        (df1["Timestamp"] >= min_time)
                                        &
                                        (df1["Timestamp"] <= max_time)
                                    ).sum()

                                    len2 = (
                                        (df2["Timestamp"] >= min_time)
                                        &
                                        (df2["Timestamp"] <= max_time)
                                    ).sum()

                                    num_points = min(
                                        len1,
                                        len2
                                    )

                                    if num_points < 2:
                                        user_results[user].append(
                                            (np.nan, np.nan)
                                        )
                                        continue

                                common_time = np.linspace(
                                    min_time,
                                    max_time,
                                    num_points
                                )

                                f1 = interp1d(
                                    df1["Timestamp"],
                                    df1[df1.columns[1]],
                                    kind="nearest",
                                    bounds_error=False,
                                    fill_value="extrapolate"
                                )

                                f2 = interp1d(
                                    df2["Timestamp"],
                                    df2[df2.columns[1]],
                                    kind="nearest",
                                    bounds_error=False,
                                    fill_value="extrapolate"
                                )

                                aligned1 = f1(common_time)
                                aligned2 = f2(common_time)

                                if (
                                    np.all(np.isnan(aligned1))
                                    or
                                    np.all(np.isnan(aligned2))
                                ):
                                    user_results[user].append(
                                        (np.nan, np.nan)
                                    )
                                    continue

                                r, p = pearsonr(
                                    aligned1,
                                    aligned2
                                )

                                user_results[user].append(
                                    (r, p)
                                )

                            except Exception as e:
                                print(
                                    f"[ERROR] Error processing "
                                    f"{dev1}-{dev2} for user "
                                    f"{user}: {e}"
                                )

                                user_results[user].append(
                                    (np.nan, np.nan)
                                )

                    technique_key = " -> ".join(
                        techniques
                    )

                    techniques_comparisons[
                        technique_key
                    ] = {}

                    for idx, (dev1, dev2) in enumerate(
                        comparisons
                    ):

                        vals = [
                            user_results[u][idx][0]
                            for u in users_list
                            if not np.isnan(
                                user_results[u][idx][0]
                            )
                        ]

                        pvals = [
                            user_results[u][idx][1]
                            for u in users_list
                            if not np.isnan(
                                user_results[u][idx][1]
                            )
                        ]

                        if not vals:
                            continue

                        mean_r = np.mean(vals)

                        non_significant = np.sum(
                            np.array(pvals) > 0.05
                        )

                        comparison_label = (
                            f"{comparisons_name[dev1]} vs "
                            f"{comparisons_name[dev2]}"
                        )

                        techniques_comparisons[
                            technique_key
                        ][comparison_label] = {
                            "mean_r": mean_r,
                            "non_significant": non_significant,
                        }

                else:
                    calculate_corrs(
                        [
                            ref[technique]
                            for ref in devices_ref
                        ],
                        techniques + [technique]
                    )

            except Exception as e:
                print(
                    f"[ERROR] Error in technique "
                    f"{technique}: {e}"
                )
                continue

    calculate_corrs(signals_ref)

    selected_technique = (
        "original -> IQR -> gaussian"
        if signal == "Gsr"
        else "original -> IQR -> butterworth"
    )

    return techniques_comparisons[selected_technique]

In [ ]:
def generate_latex_table(
    data,
    output_doc,
    signal_order=None,
    test_order=None,
    dec=3
):

    if signal_order is None:
        signal_order = list(data.keys())

    if test_order is None:
        first_signal = signal_order[0]
        test_order = list(data[first_signal].keys())

    lines = []

    lines.append(
        r"\begin{tabular}{l l "
        + "c " * len(test_order)
        + "}"
    )

    lines.append(r"\toprule")

    header = " & ".join(
        [
            f"\\textbf{{{t}}}"
            for t in ["Signal", "Comparison"] + test_order
        ]
    )

    lines.append(header + r" \\")

    lines.append(r"\midrule")

    for signal in signal_order:

        comparisons = list(
            data[signal][test_order[0]].keys()
        )

        lines.append(
            f"\\multirow{{{len(comparisons) * 2 - 1}}}"
            f"{{*}}{{\\textbf{{{'EDA' if signal == 'Gsr' else 'HR'}}}}}"
        )

        for i, comp in enumerate(comparisons):

            parts = (
                comp
                .replace(":", "")
                .split("vs")
            )

            comp_str = (
                r"\makecell[l]{"
                + r" \\ / ".join(
                    [p.strip() for p in parts]
                )
                + "}"
            )

            values = []

            for test in test_order:

                mean_r = (
                    data[signal][test][comp]["mean_r"]
                )

                ns = (
                    data[signal][test][comp]["non_significant"]
                )

                val_str = (
                    f"{mean_r:.{dec}f} / {ns}"
                )

                values.append(val_str)

            line = (
                "    & "
                + comp_str
                + " & "
                + " & ".join(values)
                + r" \\"
            )

            lines.append(line)

            if i < len(comparisons) - 1:
                lines.append(r"    \addlinespace")

        lines.append(r"\midrule")

    # Replace final \midrule with \bottomrule
    lines[-1] = r"\bottomrule"

    lines.append(r"\end{tabular}")

    with open(output_doc, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

## Generate pdfs and documents

It is important to mention that when `"shimmer_empatica"` is used, this indicates that the Shimmer device was intended to be placed in the same position as the Empatica. That is, the **PPG sensor on the dorsal side** and the **GSR sensor on the volar side**. For this reason, in the second phase, the **Y-axis labels** are different for each signal.

In [ ]:
import os
import pickle


tasks = [
    ("relax_start", "relax_end", "Baseline"),
    ("squat_test_start", "squat_test_end", "Squat test"),
    ("sit_start", "sit_end", "Relax 1"),
    ("video1_start", "video1_end", "Video 1"),
    ("video1_end", "video2_start", "Relax 2"),
    ("video2_start", "video2_end", "Video 2"),
    ("full_test_start", "full_test_end", "Full test"),
]

signals = ["Gsr", "Hr"]

device_names = [
    "emotibit_volar",
    "emotibit_dorsal",
    "shimmer_wrist",
    "shimmer_fingers",
]

colors = [
    "gold",
    "teal",
    "crimson",
    "purple",
    "magenta",
    "darkred",
]

comparisons = [
    ("emotibit_volar", "emotibit_dorsal"),
    ("shimmer_wrist", "shimmer_fingers"),
    ("emotibit_volar", "shimmer_wrist"),
    ("emotibit_volar", "shimmer_fingers"),
    ("emotibit_dorsal", "shimmer_wrist"),
    ("emotibit_dorsal", "shimmer_fingers"),
]

comparisons_ticks = {
    "Gsr": [
        "EmotiBit:\nVolar vs\nDorsal",
        "Shimmer:\nVolar vs\nFingers",
        "EmotiBit:\nVolar vs\nShimmer:\nVolar",
        "EmotiBit:\nVolar vs\nShimmer:\nFingers",
        "EmotiBit:\nDorsal vs\nShimmer:\nVolar",
        "EmotiBit:\nDorsal vs\nShimmer:\nFingers",
    ],
    "Hr": [
        "EmotiBit:\nVolar vs\nDorsal",
        "Shimmer:\nDorsal vs\nFingers",
        "EmotiBit:\nVolar vs\nShimmer:\nDorsal",
        "EmotiBit:\nVolar vs\nShimmer:\nFingers",
        "EmotiBit:\nDorsal vs\nShimmer:\nDorsal",
        "EmotiBit:\nDorsal vs\nShimmer:\nFingers",
    ],
}

comparisons_names_table = {
    "Gsr": {
        "emotibit_volar": "EmotiBit: Volar",
        "emotibit_dorsal": "EmotiBit: Dorsal",
        "shimmer_wrist": "Shimmer: Volar",
        "shimmer_fingers": "Shimmer: Fingers",
    },
    "Hr": {
        "emotibit_volar": "EmotiBit: Volar",
        "emotibit_dorsal": "EmotiBit: Dorsal",
        "shimmer_wrist": "Shimmer: Dorsal",
        "shimmer_fingers": "Shimmer: Fingers",
    },
}

# Load preprocessed data
with open(
    "processed_data/processed_data.pickle",
    "rb"
) as file:
    data = pickle.load(file)


# Output directories
table_dir = "results/table"
scatter_base_dir = "results/scatter"
n_means_dir = "results/n_means"

os.makedirs(table_dir, exist_ok=True)
os.makedirs(scatter_base_dir, exist_ok=True)
os.makedirs(n_means_dir, exist_ok=True)

# Correlation analyses
table = {}

for signal in signals:

    table[signal] = {}

    comparison_tick = comparisons_ticks[signal]
    comparisons_name = comparisons_names_table[signal]

    scatter_dir = os.path.join(
        scatter_base_dir,
        signal
    )

    os.makedirs(
        scatter_dir,
        exist_ok=True
    )

    # Pearson correlation plots for each task
    for task in tasks:

        plot_scatter_pearson(
            data=data,
            signal=signal,
            device_names=device_names,
            comparisons=comparisons,
            comparisons_tick=comparison_tick,
            colors=colors,
            output_pdf=os.path.join(
                scatter_dir,
                f"{task[2]}.pdf"
            ),
            task=task
        )

    # Correlation values for the LaTeX table
    for task in tasks:

        table[signal][task[2]] = get_table_data(
            data=data,
            signal=signal,
            device_names=device_names,
            comparisons=comparisons,
            comparisons_name=comparisons_name,
            crop=(
                task[0],
                task[1]
            )
        )


# Physiological behaviour figure
# ONE figure containing both EDA and HR
behaviour_tasks = [
    task
    for task in tasks
    if task[2] != "Full test"
]

plot_behaviour(
    data=data,
    device_names=device_names,
    tasks=behaviour_tasks,
    output_pdf=os.path.join(
        n_means_dir,
        "physiological_behaviour.pdf"
    ),
    output_png=os.path.join(
        n_means_dir,
        "physiological_behaviour.png"
    ),
    n_means=8,
    n_boot=5000,
    bootstrap_seed=2026
)


# Final LaTeX table
generate_latex_table(
    table,
    output_doc=os.path.join(
        table_dir,
        "correlation_table.txt"
    )
)

## Generate pdfs for Latex

### Scatter

In [ ]:
import os
from PyPDF2 import PdfReader, PdfWriter


pdf_files = {
    "gsr_full_test": os.path.join(
        "results",
        "scatter",
        "Gsr",
        "Full test.pdf"
    ),
    "hr_full_test": os.path.join(
        "results",
        "scatter",
        "Hr",
        "Full test.pdf"
    ),
}


output_path = os.path.join(
    "results",
    "scatter",
    "latex_pdfs"
)

os.makedirs(
    output_path,
    exist_ok=True
)


for key, pdf_path in pdf_files.items():

    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)

    # Select the fourth-to-last page generated by plot_scatter_pearson
    page_index = total_pages - 4

    if page_index < 0:
        raise ValueError(
            f"The PDF {pdf_path} does not have enough pages."
        )

    writer = PdfWriter()
    writer.add_page(
        reader.pages[page_index]
    )

    output_file = os.path.join(
        output_path,
        f"{key}.pdf"
    )

    with open(output_file, "wb") as f_out:
        writer.write(f_out)

    print(
        f"{key} saved to {output_file}"
    )

### Behavior

In [ ]:
import os
import fitz  # PyMuPDF


MM_TO_PT = 72.0 / 25.4  # ≈ 2.83465


def apply_trim_to_page(page, trim_mm):

    L_mm, B_mm, R_mm, T_mm = trim_mm

    L = float(L_mm) * MM_TO_PT
    B = float(B_mm) * MM_TO_PT
    R = float(R_mm) * MM_TO_PT
    T = float(T_mm) * MM_TO_PT

    rect = page.rect
    new_rect = fitz.Rect(rect)

    new_rect.x0 += L
    new_rect.y0 += B
    new_rect.x1 -= R
    new_rect.y1 -= T

    new_rect.x0 = max(new_rect.x0, rect.x0)
    new_rect.y0 = max(new_rect.y0, rect.y0)
    new_rect.x1 = min(new_rect.x1, rect.x1)
    new_rect.y1 = min(new_rect.y1, rect.y1)

    if new_rect.width <= 1 or new_rect.height <= 1:
        raise ValueError(
            "Invalid trim. Check the measurements."
        )

    return new_rect


def crop_pdf(
    input_path,
    out_suffix_map,
    out_dir
):

    if not os.path.isfile(input_path):
        print(
            f"[WARNING] Does not exist: {input_path}"
        )
        return

    doc = fitz.open(input_path)
    total_pages = len(doc)

    # Select the page containing the final behaviour plot
    if total_pages == 1:
        page_index = 0

    elif total_pages == 2:
        page_index = 1

    else:
        page_index = total_pages - 2

    page = doc[page_index]

    base, _ = os.path.splitext(
        os.path.basename(input_path)
    )

    base = base.lower()

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    for name, trim_mm in out_suffix_map.items():

        try:
            new_rect = apply_trim_to_page(
                page,
                trim_mm
            )

            out_doc = fitz.open()

            out_page = out_doc.new_page(
                width=new_rect.width,
                height=new_rect.height
            )

            out_page.show_pdf_page(
                out_page.rect,
                doc,
                page_index,
                clip=new_rect
            )

            # Final name:
            # <signal>_behaviour_<device>.pdf
            final_name = (
                f"{base}_{name}.pdf"
            )

            out_path = os.path.join(
                out_dir,
                final_name
            )

            out_doc.save(out_path)
            out_doc.close()

            print(
                f"[OK] Saved: {out_path}"
            )

        except Exception as e:
            print(f"[ERROR] {e}")

    doc.close()


# Crop configuration for the placement-focused data
CROPS = {
    "emotibit_volar": (
        1.0,
        1,
        174,
        1.7
    ),
    "emotibit_dorsal": (
        58.9,
        1,
        116.1,
        1.7
    ),
    "shimmer_wrist": (
        116.8,
        1,
        58.2,
        1.7
    ),
    "shimmer_fingers": (
        174.7,
        1,
        0.3,
        1.7
    ),
}


# Input / output paths
INPUTS = [
    (
        "./results/n_means/Gsr_behaviour.pdf",
        CROPS,
        "./results/n_means/latex_pdfs"
    ),
    (
        "./results/n_means/Hr_behaviour.pdf",
        CROPS,
        "./results/n_means/latex_pdfs"
    ),
]


# Process PDFs
for input_path, crops, out_dir in INPUTS:

    crop_pdf(
        input_path,
        crops,
        out_dir
    )